# Step 2 -- Pressure events (per player) on the named tracking

Reads the Step 1 tables (`player_frame_table_named`, `ball_frame_table_named`, `match_meta_named.json`).

**Pressure definition = your v30 definition** (same thresholds), now per *named* player and direction-safe (`x_att`):
a defender pressures the carrier when he is within `PRESSURE_DIST_M` (5 m) **and** (a) closes in at >= 2.5 m/s, or
(b) is touch-tight (< 1.5 m), or (c) has stepped >= 4 m ahead of his team's out-of-possession line. Closing speed is the
*relative* velocity along the defender->carrier line (smoothed velocities, no frame-window artefacts).

Then:
1. frame-level pressing rows -> **player pressure events** (sustained >= 8 frames, one carrier, short gaps bridged)
2. event features: duration, start / min distance, closing & approach speed, trigger, zone, own box, co-pressers,
   **counter-press** (starts <= 5 s after his team lost the ball)
3. **outcomes** within 2 s after the event: regain by presser / by teammate, forced pass (forward / sideways /
   backward, bypassed the presser or not), carrier retained, loose ball, ball out / stoppage
4. **team press sequences** (overlapping player events on the same carrier possession merged)
5. **per-player** pressing table + **pressed-as-carrier** (press resistance) table + team summary
6. a **review sample** (CSV with timestamps) for the Step 3 validation -- the clips are cut from it

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

CACHE_DIR = r"C:\Users\user\Desktop\Football_cv_project\football-pipeline (1)\project\barca_atletico_first_half\new_cache"
PLAYER_TABLE = os.path.join(CACHE_DIR, "player_frame_table_named.parquet")
BALL_TABLE   = os.path.join(CACHE_DIR, "ball_frame_table_named.parquet")
META_PATH    = os.path.join(CACHE_DIR, "match_meta_named.json")
OUT_DIR      = os.path.join(CACHE_DIR, "pressure")

# ---- v30 pressure definition (unchanged) ----
PRESSURE_DIST_M = 5.0
CLOSING_SPEED_THRESH_MPS = 2.5
TIGHT_MARK_DIST_M = 1.5
STEP_OUT_MARGIN_M = 4.0
LINE_HEIGHT_DEFENDERS_N = 3
MIN_SUSTAINED_PRESSURE_FRAMES = 8     # v30 fix: a press must last >= 8 frames (no single-frame flicker)
# ---- event building ----
BRIDGE_GAP_FRAMES = 3                 # a press interrupted for <= 3 frames is the same press
PRESSERS_ROLES = ("player",)          # goalkeepers are not counted as pressers (as in v30)
COUNTER_PRESS_WINDOW_S = 5.0
OUTCOME_WINDOW_S = 2.0                # what happens up to 2 s after the press ends
PROGRESS_THRESH_M = 3.0               # forced pass forward / backward threshold (in the passer's attack direction)
BYPASS_MARGIN_M = 2.0
BOX_DEPTH_M, BOX_HALF_WIDTH_M = 16.5, 20.16
REVIEW_SAMPLE_N = 80

os.makedirs(OUT_DIR, exist_ok=True)
pf = pd.read_parquet(PLAYER_TABLE)
bt = pd.read_parquet(BALL_TABLE).set_index("frame_idx").sort_index()
meta = json.load(open(META_PATH, encoding="utf-8"))
FPS, L, W = meta["fps"], meta["pitch_length"], meta["pitch_width"]
TEAM_NAMES = {int(k): v for k, v in meta["team_names"].items()}
ATTACK_DIR = {int(k): v for k, v in meta["attack_dir"].items()}
PEOPLE = {int(k): v for k, v in meta["people"].items()}
NAME = {t: f"{v['team_name'] or ''} {v['number'] if v['number'] is not None else ''} {v['name']}".strip() for t, v in PEOPLE.items()}
N_FRAMES = meta["n_frames"]
OUTCOME_WIN = int(OUTCOME_WINDOW_S * FPS)
print(f"player rows {len(pf):,} | frames {N_FRAMES} | teams {TEAM_NAMES} | attack dir {ATTACK_DIR}")

## 0. Team out-of-possession line (for the "stepped out" rule)
Mean depth (`x_att`) of each team's last 3 outfield players, only in frames where that team does not have the ball.

In [ ]:
live = pf[~pf.is_stoppage & pf.pitch_x.notna()]
oop = live[(live.role == "player") & live.possession_team.notna() & (live.team != live.possession_team)]
TEAM_LINE = (oop.sort_values("x_att").groupby(["frame_idx", "team"]).head(LINE_HEIGHT_DEFENDERS_N)
             .groupby("team").x_att.mean().to_dict())
TEAM_LINE = {int(k): float(v) for k, v in TEAM_LINE.items()}
for k, v in TEAM_LINE.items():
    print(f"{TEAM_NAMES[k]}: out-of-possession line {v:.1f} m from own goal -> 'stepped out' = x_att >= {v + STEP_OUT_MARGIN_M:.1f} m")

## 1. Frame-level pressure: every defender vs the carrier, every live frame

In [ ]:
car = live[live.is_carrier][["frame_idx", "track_id", "team", "pitch_x", "pitch_y", "vx", "vy", "x_att"]]
car = car.drop_duplicates("frame_idx").rename(columns={"track_id": "carrier_id", "team": "carrier_team", "pitch_x": "cx",
                                                       "pitch_y": "cy", "vx": "cvx", "vy": "cvy", "x_att": "carrier_x_att"})
dfd = live[live.role.isin(PRESSERS_ROLES)][["frame_idx", "track_id", "team", "pitch_x", "pitch_y", "vx", "vy", "speed_mps",
                                            "x_att", "source"]]
fr = dfd.merge(car, on="frame_idx", how="inner")
fr = fr[fr.team.notna() & fr.carrier_team.notna() & (fr.team != fr.carrier_team)].copy()

dx, dy = fr.cx - fr.pitch_x, fr.cy - fr.pitch_y
fr["dist"] = np.hypot(dx, dy)
ux, uy = dx / fr.dist.where(fr.dist > 0.05), dy / fr.dist.where(fr.dist > 0.05)
fr["closing_mps"] = (fr.vx - fr.cvx) * ux + (fr.vy - fr.cvy) * uy        # + = gap to the carrier shrinking
fr["approach_mps"] = fr.vx * ux + fr.vy * uy                              # defender's own speed towards the carrier
fr["stepped_out"] = fr.x_att >= fr.team.map(lambda k: TEAM_LINE.get(int(k), np.inf)).astype(float) + STEP_OUT_MARGIN_M
near = fr.dist <= PRESSURE_DIST_M
fr["trig_closing"] = near & (fr.closing_mps >= CLOSING_SPEED_THRESH_MPS)
fr["trig_tight"] = fr.dist <= TIGHT_MARK_DIST_M
fr["trig_stepout"] = near & fr.stepped_out
fr["pressing"] = fr.trig_closing | fr.trig_tight | fr.trig_stepout
press_rows = fr[fr.pressing].sort_values(["track_id", "frame_idx"]).reset_index(drop=True)
print(f"defender-vs-carrier rows: {len(fr):,} | pressing rows: {len(press_rows):,} "
      f"({press_rows.frame_idx.nunique():,} frames with >= 1 presser)")
print("trigger share among pressing rows:",
      {k: f"{press_rows[k].mean():.0%}" for k in ("trig_closing", "trig_tight", "trig_stepout")})

## 2. Player pressure events

In [ ]:
p = press_rows
new = np.r_[True, (p.track_id.values[1:] != p.track_id.values[:-1]) |
                  (p.carrier_id.values[1:] != p.carrier_id.values[:-1]) |
                  (np.diff(p.frame_idx.values) > BRIDGE_GAP_FRAMES + 1)]
p["ev"] = np.cumsum(new) - 1
first = p.groupby("ev").head(1).set_index("ev")
ev = p.groupby("ev").agg(start=("frame_idx", "min"), end=("frame_idx", "max"), n_press_frames=("frame_idx", "size"),
                         presser=("track_id", "first"), team=("team", "first"), carrier=("carrier_id", "first"),
                         carrier_team=("carrier_team", "first"), min_dist=("dist", "min"),
                         mean_closing=("closing_mps", "mean"), max_closing=("closing_mps", "max"),
                         max_approach=("approach_mps", "max"), max_speed=("speed_mps", "max"),
                         share_closing=("trig_closing", "mean"), share_tight=("trig_tight", "mean"),
                         share_stepout=("trig_stepout", "mean"), est_frames=("source", lambda s: (s == "interpolated").mean()))
ev = ev[ev.n_press_frames >= MIN_SUSTAINED_PRESSURE_FRAMES].copy()
ev["start_dist"] = first.loc[ev.index, "dist"]
ev["press_x"], ev["press_y"] = first.loc[ev.index, "pitch_x"], first.loc[ev.index, "pitch_y"]
ev["press_x_att"] = first.loc[ev.index, "x_att"]                     # presser's depth in HIS attack direction
ev["zone"] = pd.cut(ev.press_x_att, [-np.inf, 35, 70, np.inf], labels=["own_third", "middle_third", "attacking_third"])
ev["in_own_box"] = (ev.press_x_att <= BOX_DEPTH_M) & ((ev.press_y - W / 2).abs() <= BOX_HALF_WIDTH_M)
ev["duration_s"] = (ev.end - ev.start + 1) / FPS
# main trigger, by priority: an active run counts first, then touch-tight marking, then just "stepped out"
ev["trigger"] = np.where(ev.share_closing >= 0.3, "closing", np.where(ev.share_tight >= 0.3, "tight", "stepout"))
ev["team"] = ev.team.astype(int); ev["carrier_team"] = ev.carrier_team.astype(int)
ev = ev.reset_index(drop=True); ev.index.name = "event_id"

# co-pressers: other defenders pressing the SAME carrier during the event
by_frame = p.groupby(["frame_idx", "carrier_id"]).track_id.apply(set).to_dict()
co, n_max = [], []
for e in ev.itertuples():
    s, mx = set(), 1
    for f in range(e.start, e.end + 1):
        q = by_frame.get((f, e.carrier), set())
        s |= q; mx = max(mx, len(q))
    s.discard(e.presser); co.append(sorted(s)); n_max.append(mx)
ev["co_pressers"] = co
ev["n_co_pressers"] = [len(c) for c in co]
ev["max_simultaneous"] = n_max

# counter-press: the presser's team lost possession <= COUNTER_PRESS_WINDOW_S before the press started
poss = bt.possession_team.astype("float").reindex(range(N_FRAMES)).values
chg = np.where((~np.isnan(poss[1:])) & (~np.isnan(poss[:-1])) & (poss[1:] != poss[:-1]))[0] + 1
loss_frames = {k: chg[poss[chg - 1] == k] for k in TEAM_NAMES}
def last_loss(team, f):
    a = loss_frames[team]; i = np.searchsorted(a, f, side="right") - 1
    return a[i] if i >= 0 else None
ev["since_loss_s"] = [(e.start - lf) / FPS if (lf := last_loss(e.team, e.start)) is not None else np.nan for e in ev.itertuples()]
ev["counter_press"] = ev.since_loss_s <= COUNTER_PRESS_WINDOW_S
print(f"player pressure events: {len(ev)} | by team: {ev.team.map(TEAM_NAMES).value_counts().to_dict()}")
print(ev[["duration_s", "start_dist", "min_dist", "mean_closing", "max_approach", "n_co_pressers"]].describe().round(2).to_string())

## 3. Outcomes
Read from the carrier sequence after the press starts, up to `OUTCOME_WINDOW_S` after it ends. The first thing that
happens decides: the ball goes to the pressing team (**regain**, by the presser or a teammate), to a teammate of the
carrier (**forced pass**: forward / sideways / backward, and whether it went past the presser), nobody
(**loose ball**), dead ball (**stoppage**), or the carrier still has it (**retained** = press beaten).

In [ ]:
carr = bt.carrier_track_id.astype("float").reindex(range(N_FRAMES)).values
cteam = bt.carrier_team.astype("float").reindex(range(N_FRAMES)).values
stop = bt.is_stoppage.reindex(range(N_FRAMES)).fillna(False).values.astype(bool)
pos_idx = pf.set_index(["frame_idx", "track_id"])[["pitch_x", "pitch_y", "x_att"]]

def xpos(f, t):
    try:
        r = pos_idx.loc[(int(f), int(t))]
        return (float(r.pitch_x), float(r.pitch_y)) if np.isfinite(r.pitch_x) else None
    except KeyError:
        return None

def along(x, team):   # pitch x expressed in `team`'s attack direction (0 = own goal)
    return x if ATTACK_DIR[team] == 1 else L - x

def outcome(start, end, carrier, carrier_team, pressers, presser):
    stop_at = min(N_FRAMES - 1, end + OUTCOME_WIN)
    for f in range(start + 1, stop_at + 1):
        if stop[f]:
            return dict(outcome="stoppage", outcome_frame=f)
        c = carr[f]
        if np.isnan(c) or c == carrier:
            continue
        c, ct = int(c), int(cteam[f])
        if ct != carrier_team:
            return dict(outcome="regain_by_presser" if c in pressers else "regain_by_teammate",
                        outcome_frame=f, new_carrier=c, time_to_regain_s=(f - start) / FPS)
        # forced pass: release = last frame the carrier had it
        rel = f - 1
        while rel > start and carr[rel] != carrier:
            rel -= 1
        a, b, pz = xpos(rel, carrier), xpos(f, c), xpos(rel, presser)
        res = dict(outcome="forced_pass", outcome_frame=f, new_carrier=c)
        if a and b:
            prog = along(b[0], carrier_team) - along(a[0], carrier_team)
            res["pass_progress_m"] = prog
            res["pass_dir"] = "forward" if prog > PROGRESS_THRESH_M else ("backward" if prog < -PROGRESS_THRESH_M else "sideways")
            if pz:
                res["bypassed_presser"] = along(b[0], carrier_team) > along(pz[0], carrier_team) + BYPASS_MARGIN_M
        return res
    if not np.isnan(carr[stop_at]) and carr[stop_at] == carrier:
        return dict(outcome="retained")
    return dict(outcome="loose_ball")

out = [outcome(e.start, e.end, e.carrier, e.carrier_team, set([e.presser] + e.co_pressers), e.presser) for e in ev.itertuples()]
ev = ev.join(pd.DataFrame(out, index=ev.index))
ev["regain"] = ev.outcome.str.startswith("regain")
ev["disrupted"] = ev.regain | (ev.outcome == "loose_ball") | (ev.pass_dir == "backward")
ev["press_beaten"] = (ev.outcome == "retained") | (ev.bypassed_presser == True)
ev["presser_name"] = ev.presser.map(NAME); ev["carrier_name"] = ev.carrier.map(NAME)
ev["team_name"] = ev.team.map(TEAM_NAMES)
ev["t_start"] = ev.start.map(lambda f: f"{int(f / FPS // 60):02d}:{f / FPS % 60:04.1f}")
print(pd.crosstab(ev.team_name, ev.outcome, margins=True).to_string())
print("\nforced pass direction:", ev.pass_dir.value_counts().to_dict())

## 4. Team press sequences (overlapping player presses on the same carrier possession)

In [ ]:
seqs = []
for team, g in ev.sort_values("start").groupby("team"):
    cur = None
    for e in g.itertuples():
        if cur and e.start <= cur["end"] + BRIDGE_GAP_FRAMES and e.carrier_team == cur["carrier_team"] and \
                not (poss[cur["end"]:e.start + 1] == team).any():
            cur["end"] = max(cur["end"], e.end); cur["events"].append(e.Index); cur["pressers"].add(e.presser)
            cur["carriers"].add(e.carrier)
        else:
            if cur: seqs.append(cur)
            cur = dict(team=team, carrier_team=e.carrier_team, start=e.start, end=e.end, events=[e.Index],
                       pressers={e.presser}, carriers={e.carrier}, first_carrier=e.carrier, first_presser=e.presser,
                       zone=e.zone, press_x_att=e.press_x_att, counter_press=e.counter_press)
    if cur: seqs.append(cur)
sq = pd.DataFrame(seqs)
sq_out = [outcome(r.start, r.end, r.first_carrier, r.carrier_team, r.pressers, r.first_presser) for r in sq.itertuples()]
sq = sq.join(pd.DataFrame(sq_out))
sq["n_pressers"] = sq.pressers.map(len)
sq["duration_s"] = (sq.end - sq.start + 1) / FPS
sq["regain"] = sq.outcome.str.startswith("regain")
sq["team_name"] = sq.team.map(TEAM_NAMES)
sq.index.name = "sequence_id"
print(f"press sequences: {len(sq)} | by team {sq.team_name.value_counts().to_dict()}")
print(sq.groupby("team_name").agg(regain_rate=("regain", "mean"), mean_pressers=("n_pressers", "mean"),
                                  mean_duration_s=("duration_s", "mean"), counter_press_share=("counter_press", "mean")).round(3).to_string())

## 5. Per-player pressing table, pressed-as-carrier table, team summary
Rates per **visible live minute** (the camera does not see everybody all the time). `opp_poss_visible_min` = minutes the
player was on screen while the opponent had the ball -- the fair denominator for pressing.

In [ ]:
lv = pf[~pf.is_stoppage]
vis_min = lv.groupby("track_id").size() / FPS / 60
opp_vis_min = lv[lv.possession_team.notna() & (lv.team != lv.possession_team)].groupby("track_id").size() / FPS / 60

g = ev.groupby("presser")
P = pd.DataFrame({
    "player": [NAME[t] for t in g.size().index],
    "team": [PEOPLE[t]["team_name"] for t in g.size().index],
    "pressures": g.size(),
    "pressing_time_s": g.duration_s.sum(),
    "visible_min": vis_min.reindex(g.size().index),
    "opp_poss_visible_min": opp_vis_min.reindex(g.size().index),
    "regains": g.regain.sum(),
    "regains_himself": g.outcome.apply(lambda s: (s == "regain_by_presser").sum()),
    "regain_rate": g.regain.mean(),
    "disrupt_rate": g.disrupted.mean(),
    "beaten_rate": g.press_beaten.mean(),
    "forced_back_pass": g.pass_dir.apply(lambda s: (s == "backward").sum()),
    "high_press_share": g.zone.apply(lambda s: (s == "attacking_third").mean()),
    "counter_presses": g.counter_press.sum(),
    "co_press_share": g.n_co_pressers.apply(lambda s: (s > 0).mean()),
    "mean_closing_mps": g.mean_closing.mean(),
    "max_approach_mps": g.max_approach.max(),
    "mean_min_dist_m": g.min_dist.mean(),
})
P["pressures_per_10_opp_min"] = P.pressures / P.opp_poss_visible_min * 10
partners = ev.explode("co_pressers").dropna(subset=["co_pressers"]).groupby("presser").co_pressers.agg(
    lambda s: Counter(s).most_common(1)[0])
P["top_partner"] = [f"{NAME[p_[0]]} ({p_[1]})" if isinstance(p_, tuple) else "" for p_ in partners.reindex(P.index)]
P = P.sort_values(["team", "pressures_per_10_opp_min"], ascending=[True, False])
P.index.name = "track_id"
pd.set_option("display.width", 220)
print(P.round(2).to_string())

gc = ev.groupby("carrier")
R = pd.DataFrame({"player": [NAME[t] for t in gc.size().index], "team": [PEOPLE[t]["team_name"] for t in gc.size().index],
                  "times_pressed": gc.size(),
                  "lost_ball": gc.regain.sum(),
                  "kept_or_passed": gc.outcome.apply(lambda s: s.isin(["retained", "forced_pass"]).sum()),
                  "forward_pass_under_press": gc.pass_dir.apply(lambda s: (s == "forward").sum())})
R["press_resistance"] = R.kept_or_passed / R.times_pressed
R = R.sort_values(["team", "times_pressed"], ascending=[True, False]); R.index.name = "track_id"
print("\npressed as carrier (press resistance):")
print(R.round(2).to_string())

team_summary = {}
for k, nm in TEAM_NAMES.items():
    e, s = ev[ev.team == k], sq[sq.team == k]
    opp_poss_min = float(((bt.possession_team == (1 - k)) & ~bt.is_stoppage).sum() / FPS / 60)
    team_summary[nm] = {
        "player_pressures": int(len(e)), "press_sequences": int(len(s)),
        "sequences_per_opp_possession_min": round(len(s) / max(opp_poss_min, 1e-9), 2),
        "sequence_regain_rate": round(float(s.regain.mean()), 3) if len(s) else None,
        "player_press_regain_rate": round(float(e.regain.mean()), 3) if len(e) else None,
        "mean_pressers_per_sequence": round(float(s.n_pressers.mean()), 2) if len(s) else None,
        "median_press_height_x_att": round(float(e.press_x_att.median()), 1) if len(e) else None,
        "zone_share": e.zone.value_counts(normalize=True).round(3).to_dict(),
        "counter_press_share": round(float(e.counter_press.mean()), 3) if len(e) else None,
        "forced_back_passes": int((e.pass_dir == "backward").sum()),
    }
print("\nteam summary:", json.dumps(team_summary, indent=1, default=str))

## 6. Review sample for validation (Step 3) + save
A stratified sample (every outcome, every team, both short and long presses) with timestamps. Fill the empty
`label_*` columns while watching the clips (next step cuts them automatically from this CSV).

In [ ]:
rng = np.random.default_rng(0)
strata = ev.groupby(["team_name", "outcome"])
per = max(1, REVIEW_SAMPLE_N // max(strata.ngroups, 1))
smp = pd.concat([g_.sample(min(len(g_), per), random_state=0) for _, g_ in strata])
if len(smp) < REVIEW_SAMPLE_N:
    rest = ev.drop(smp.index)
    smp = pd.concat([smp, rest.sample(min(len(rest), REVIEW_SAMPLE_N - len(smp)), random_state=1)])
review = smp.sort_values("start")[["t_start", "start", "end", "duration_s", "team_name", "presser_name", "carrier_name",
                                    "co_pressers", "outcome", "pass_dir", "min_dist", "mean_closing", "trigger", "zone"]].copy()
review["co_pressers"] = review.co_pressers.map(lambda c: ", ".join(NAME[t] for t in c))
for c in ("label_is_pressure", "label_presser_correct", "label_outcome_correct", "notes"):
    review[c] = ""
review.to_csv(os.path.join(OUT_DIR, "pressure_review_sample.csv"), encoding="utf-8-sig")

ev_out = ev.copy(); ev_out["co_pressers"] = ev_out.co_pressers.map(lambda c: ",".join(map(str, c)))
ev_out.to_parquet(os.path.join(OUT_DIR, "pressure_events_player.parquet"))
sq_out_df = sq.copy()
for c in ("events", "pressers", "carriers"):
    sq_out_df[c] = sq_out_df[c].map(lambda v: ",".join(map(str, sorted(v))))
sq_out_df.to_parquet(os.path.join(OUT_DIR, "press_sequences.parquet"))
press_rows.to_parquet(os.path.join(OUT_DIR, "pressure_frames.parquet"), index=False)
P.to_csv(os.path.join(OUT_DIR, "player_pressing_stats.csv"), encoding="utf-8-sig")
R.to_csv(os.path.join(OUT_DIR, "player_press_resistance.csv"), encoding="utf-8-sig")
with open(os.path.join(OUT_DIR, "team_pressure_summary.json"), "w", encoding="utf-8") as fh:
    json.dump({"params": {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, tuple))},
               "team_line": TEAM_LINE, "teams": team_summary}, fh, indent=1, default=str)
print(f"Saved to {OUT_DIR}: pressure_events_player ({len(ev)}), press_sequences ({len(sq)}), pressure_frames, "
      f"player_pressing_stats, player_press_resistance, team_pressure_summary, pressure_review_sample ({len(review)})")

fig, axs = plt.subplots(1, 2, figsize=(15, 6))
for ax, (col, ttl) in zip(axs, [("pressures_per_10_opp_min", "pressures per 10 visible min of opponent possession"),
                                ("regain_rate", "regain rate of his pressures")]):
    d = P.sort_values(col)
    ax.barh(d.player, d[col], color=[("tab:red" if t == "ATM" else "tab:blue") for t in d.team])
    ax.set_title(ttl); ax.tick_params(axis="y", labelsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "player_pressing_overview.png"), dpi=110); plt.show()